In [16]:
import pandas as pd
import json
import os
from tqdm import tqdm
from glob import glob

In [17]:
class ner_eval():
    def __init__(self, true_list, pred_list, eval_dict, inv=False, sentence_eval_dict=None):
        """
        Initialize ner_eval class for both entity-level and sentence-level evaluation.
        """
        self.true_list = true_list
        self.pred_list = pred_list
        self.eval_dict = eval_dict  # For strict/relax entity-level evaluation
        self.inv = inv  # Whether to reverse true and pred lists
        self.true_length = len(true_list)
        self.pred_length = len(pred_list)
        self.check = self.true_length == self.pred_length  # Validate input lengths
        self.set_info()  # Process BIO tags to get entity spans
        
        # Initialize sentence-level evaluation dictionary
        self.sentence_eval_dict = sentence_eval_dict if sentence_eval_dict is not None else {}

    def get_info(self):
        """
        Extract entity spans (start idx, end idx, entity type) from BIO tags.
        """
        s_idx_list = []
        e_idx_list = []
        ent_list = []
        true_list = self.pred_list if self.inv else self.true_list
        if self.check:
            cnt = -1
            for i in range(len(true_list)):
                if true_list[i] != 'O':
                    label = true_list[i].split('-')
                    if label[0] == 'B':
                        cnt += 1
                        s_idx_list.append(i)
                        e_idx_list.append(i+1)
                        ent_list.append(label[1])
                    elif label[0] == 'I':
                        try:
                            prev = true_list[i-1]
                            if prev == 'O' or prev.split('-')[1] != label[1]:
                                cnt += 1
                                s_idx_list.append(i)
                                e_idx_list.append(i+1)
                                ent_list.append(label[1])
                            else:
                                e_idx_list[cnt] = i+1
                        except:
                            cnt += 1
                            s_idx_list.append(i)
                            e_idx_list.append(i+1)
                            ent_list.append(label[1])
        return s_idx_list, e_idx_list, ent_list

    def set_info(self):
        """
        Save entity spans.
        """
        if self.check:
            self.s_idx_list, self.e_idx_list, self.ent_list = self.get_info()

    def strict(self, true, pred, s_idx, e_idx, entity):
        """
        Strict match: entity spans and labels must match exactly.
        """
        if true[s_idx] != f'B-{entity}' or pred[s_idx] != f'B-{entity}':
            return False
        for idx in range(s_idx, e_idx):
            if true[idx] != pred[idx]:
                return False
        if e_idx < len(true) and (true[e_idx] == f'I-{entity}' or pred[e_idx] == f'I-{entity}'):
            return False
        return True

    def relax(self, true, pred, s_idx, e_idx, entity):
        """
        Relaxed match: any token overlap of the same entity type counts as correct.
        """
        for idx in range(s_idx, e_idx):
            try:
                true_ent = true[idx].split('-')[1]
                pred_ent = pred[idx].split('-')[1]
                if true_ent == pred_ent == entity:
                    return True
            except:
                continue
        return False        

    def get_strict(self, s_idx, e_idx, ent):
        return self.strict(self.pred_list, self.true_list, s_idx, e_idx, ent) if self.inv else self.strict(self.true_list, self.pred_list, s_idx, e_idx, ent)

    def get_relax(self, s_idx, e_idx, ent):
        return self.relax(self.pred_list, self.true_list, s_idx, e_idx, ent) if self.inv else self.relax(self.true_list, self.pred_list, s_idx, e_idx, ent)

    def sentence_eval(self):
        """
        Entity-level evaluation per sentence (strict and relax counts).
        """
        result = {}
        for i in range(len(self.s_idx_list)):
            s_idx = self.s_idx_list[i]
            e_idx = self.e_idx_list[i]
            ent = self.ent_list[i]
            strict = self.get_strict(s_idx, e_idx, ent)
            relax = self.get_relax(s_idx, e_idx, ent)
            if ent in result.keys():
                result[ent]['strict'] += 1 if strict else 0
                result[ent]['relax'] += 1 if relax else 0
                result[ent]['total'] += 1
            else:
                result[ent] = {'strict': int(strict), 'relax': int(relax), 'total': 1}
        return result

    def update_dict(self):
        """
        Update overall entity-level evaluation dictionary.
        """
        if self.check:
            eval_dict = self.eval_dict
            result = self.sentence_eval()
            for k, v in result.items():
                if k in eval_dict.keys():
                    eval_dict[k]['strict'] += v['strict']
                    eval_dict[k]['relax'] += v['relax']
                    eval_dict[k]['total'] += v['total']
                else:
                    eval_dict[k] = v
            return eval_dict
        else:
            return self.eval_dict

    def sum_and_fulfilldict(self):
        """
        Fill in missing categories and compute overall counts.
        """
        eval_dict = self.eval_dict
        targeted_category = ['Adherence','Concern','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma']
        missing_category = list(set(targeted_category) - set(eval_dict.keys()))
        for i in missing_category:
            eval_dict[i] = {'strict': 0, 'relax': 0, 'total': 0}
        strict, relax, total = 0, 0, 0
        for v in eval_dict.values():
            strict += v['strict']
            relax += v['relax']
            total += v['total']
        eval_dict['overall'] = {'strict': strict, 'relax': relax, 'total': total}
        return dict(sorted(eval_dict.items()))

    ### ------------------- Sentence-Level Evaluation ------------------- ###
    def update_sentence_level_dict(self):
        """
        Update sentence-level TP, FP, FN counts per category.
        """
        if not self.check:
            return self.sentence_eval_dict
        
        true_entities = set()
        pred_entities = set()

        for tag in self.true_list:
            if tag.startswith('B-') or tag.startswith('I-'):
                true_entities.add(tag.split('-')[-1])
        for tag in self.pred_list:
            if tag.startswith('B-') or tag.startswith('I-'):
                pred_entities.add(tag.split('-')[-1])

        categories = ['Adherence','Concern','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma']

        for cat in categories:
            if cat not in self.sentence_eval_dict:
                self.sentence_eval_dict[cat] = {'TP': 0, 'FP': 0, 'FN': 0}

            if cat in true_entities and cat in pred_entities:
                self.sentence_eval_dict[cat]['TP'] += 1
            if cat not in true_entities and cat in pred_entities:
                self.sentence_eval_dict[cat]['FP'] += 1
            if cat in true_entities and cat not in pred_entities:
                self.sentence_eval_dict[cat]['FN'] += 1

        return self.sentence_eval_dict
    

    
    def compute_sentence_level_metrics(self):
        """
        Compute precision, recall, F1 for sentence-level.
        """
        categories = ['Adherence','Concern','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma']
        sentence_metrics = {}

        for cat in categories:
            if cat not in self.sentence_eval_dict:
                TP = FP = FN = 0
            else:
                TP = self.sentence_eval_dict[cat]['TP']
                FP = self.sentence_eval_dict[cat]['FP']
                FN = self.sentence_eval_dict[cat]['FN']

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

            sentence_metrics[cat] = {
                'sentence-level': {'precision': round(precision,3), 'recall': round(recall,3), 'f-1': round(f1,3)}
            }
        return sentence_metrics
    
    def compute_entity_level_metrics_separated(self, true_eval_dict, pred_eval_dict):
        metrics = {}
        categories = ['Adherence','Concern','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma']

        for cat in categories:
            # Strict
            tp_strict = true_eval_dict[cat]['strict']
            total_true = true_eval_dict[cat]['total']
            total_pred = pred_eval_dict[cat]['total']

            strict_precision = tp_strict / total_pred if total_pred > 0 else 0.0
            strict_recall = tp_strict / total_true if total_true > 0 else 0.0
            strict_f1 = (2 * strict_precision * strict_recall) / (strict_precision + strict_recall) if (strict_precision + strict_recall) > 0 else 0.0

            # Relax
            tp_relax = true_eval_dict[cat]['relax']
            relax_precision = tp_relax / total_pred if total_pred > 0 else 0.0
            relax_recall = tp_relax / total_true if total_true > 0 else 0.0
            relax_f1 = (2 * relax_precision * relax_recall) / (relax_precision + relax_recall) if (relax_precision + relax_recall) > 0 else 0.0

            metrics[cat] = {
                'strict': {'precision': round(strict_precision,3), 'recall': round(strict_recall,3), 'f-1': round(strict_f1,3)},
                'relax': {'precision': round(relax_precision,3), 'recall': round(relax_recall,3), 'f-1': round(relax_f1,3)}
            }
        return metrics
    def compute_overall_metrics_separated(self, true_eval_dict, pred_eval_dict, sentence_eval_dict):
        # ---- Entity-level Overall ----
        strict_tp_sum = 0
        strict_total_true = 0
        strict_total_pred = 0
        relax_tp_sum = 0
        relax_total_true = 0
        relax_total_pred = 0

        categories = ['Adherence','Concern','Education','Employment','Financial','Healthcare','Insurance','Literacy','Living','MentalHealth','Recommendation','Smoke','Social','SubstanceUse','Transportation','Trauma']

        for cat in categories:
            strict_tp_sum += true_eval_dict[cat]['strict']
            strict_total_true += true_eval_dict[cat]['total']
            strict_total_pred += pred_eval_dict[cat]['total']

            relax_tp_sum += true_eval_dict[cat]['relax']
            relax_total_true += true_eval_dict[cat]['total']
            relax_total_pred += pred_eval_dict[cat]['total']

        strict_precision = strict_tp_sum / strict_total_pred if strict_total_pred > 0 else 0.0
        strict_recall = strict_tp_sum / strict_total_true if strict_total_true > 0 else 0.0
        strict_f1 = (2 * strict_precision * strict_recall) / (strict_precision + strict_recall) if (strict_precision + strict_recall) > 0 else 0.0

        relax_precision = relax_tp_sum / relax_total_pred if relax_total_pred > 0 else 0.0
        relax_recall = relax_tp_sum / relax_total_true if relax_total_true > 0 else 0.0
        relax_f1 = (2 * relax_precision * relax_recall) / (relax_precision + relax_recall) if (relax_precision + relax_recall) > 0 else 0.0

        # ---- Sentence-level Overall ----
        sentence_TP_sum = 0
        sentence_FP_sum = 0
        sentence_FN_sum = 0
        for cat in categories:
            if cat in sentence_eval_dict:
                sentence_TP_sum += sentence_eval_dict[cat]['TP']
                sentence_FP_sum += sentence_eval_dict[cat]['FP']
                sentence_FN_sum += sentence_eval_dict[cat]['FN']

        sentence_precision = sentence_TP_sum / (sentence_TP_sum + sentence_FP_sum) if (sentence_TP_sum + sentence_FP_sum) > 0 else 0.0
        sentence_recall = sentence_TP_sum / (sentence_TP_sum + sentence_FN_sum) if (sentence_TP_sum + sentence_FN_sum) > 0 else 0.0
        sentence_f1 = (2 * sentence_precision * sentence_recall) / (sentence_precision + sentence_recall) if (sentence_precision + sentence_recall) > 0 else 0.0

        return {
            'strict': {'precision': round(strict_precision,3), 'recall': round(strict_recall,3), 'f-1': round(strict_f1,3)},
            'relax': {'precision': round(relax_precision,3), 'recall': round(relax_recall,3), 'f-1': round(relax_f1,3)},
            'sentence-level': {'precision': round(sentence_precision,3), 'recall': round(sentence_recall,3), 'f-1': round(sentence_f1,3)}
        }



In [18]:
def fix_bio_tags(bio_lists):
    fixed = []
    for tag_list in bio_lists:
        new_tags = []
        for i, tag in enumerate(tag_list):
            # If tag is 'O', nothing to change.
            if tag == 'O':
                new_tags.append(tag)
            else:
                prefix, entity = tag.split('-', 1)
                # If first token, or previous token is 'O', or previous token is a different entity,
                # then this token should be a beginning (B-) tag.
                if i == 0 or tag_list[i-1] == 'O' or (tag_list[i-1] != 'O' and tag_list[i-1].split('-', 1)[1] != entity):
                    new_tags.append('B-' + entity)
                else:
                    # Otherwise, continue the entity span as an inside (I-) tag.
                    new_tags.append('I-' + entity)
        fixed.append(new_tags)
    return fixed

def refine_bio_tags(bio_lists):
    refined = []
    for tags in bio_lists:
        new_tags = tags[:]  # work on a copy
        # Iterate from the second token to the second-to-last token.
        for i in range(1, len(new_tags) - 1):
            # Look for an "O" token.
            if new_tags[i] == 'O':
                prev_tag = new_tags[i - 1]
                next_tag = new_tags[i + 1]
                # Check if the previous token is part of an entity and the next token starts an entity.
                if prev_tag != 'O' and next_tag.startswith('B-'):
                    prev_entity = prev_tag.split('-', 1)[1]
                    next_entity = next_tag.split('-', 1)[1]
                    # If both tokens refer to the same entity, fill in the gap and adjust the following token.
                    if prev_entity == next_entity:
                        new_tags[i] = 'I-' + prev_entity
                        new_tags[i + 1] = 'I-' + next_entity
        refined.append(new_tags)
    return refined


In [19]:
model_results = {
    'BERT': [],
    'BioBERT': [],
    'roberta': []
}


In [20]:
model_results = {}

for model_name in ['BERT', 'BioBERT', 'roberta']:
    model_folds_metrics = []
    for fold in range(1, 6):
        for d in glob(f'../output/{model_name}_fold_{fold}*/predictions/'):
            y_true = []
            with open(f'../data/splitted_data/fold_{fold}/test.json', 'r') as textfile:
                for i in json.load(textfile):
                    y_true.append(i['ner_tags'])

            predict_file = d + '/predictions.txt'
            y_pred = []
            with open(predict_file) as txtfile:
                for i in txtfile.readlines():
                    y_pred.append(i.split())
            y_pred = refine_bio_tags(fix_bio_tags(y_pred))

            # ----- Initialize two entity dicts -----
            eval_dict_true = {}
            eval_dict_pred = {}

            # ----- Initialize sentence-level dict -----
            sentence_eval_dict = {}

            # --------- First pass: y_true reference ---------
            for i in range(len(y_true)):
                eval_dict_true = ner_eval(y_true[i], y_pred[i], eval_dict_true).update_dict()
                sentence_eval_dict = ner_eval(y_true[i], y_pred[i], eval_dict_true, sentence_eval_dict=sentence_eval_dict).update_sentence_level_dict()
            eval_dict_true = ner_eval(y_true[i], y_pred[i], eval_dict_true).sum_and_fulfilldict()

            # --------- Second pass: y_pred reference (inv=True) ---------
            for i in range(len(y_true)):
                eval_dict_pred = ner_eval(y_true[i], y_pred[i], eval_dict_pred, inv=True).update_dict()
            eval_dict_pred = ner_eval(y_true[i], y_pred[i], eval_dict_pred, inv=True).sum_and_fulfilldict()

            # --------- Compute metrics ---------
            ner_instance = ner_eval(y_true[0], y_pred[0], eval_dict_true, sentence_eval_dict=sentence_eval_dict)

            # Compute correct entity-level metrics (passing both dicts!)
            entity_metrics = ner_instance.compute_entity_level_metrics_separated(eval_dict_true, eval_dict_pred)

            # Compute sentence-level metrics
            sentence_metrics = ner_instance.compute_sentence_level_metrics()

            # --------- Combine strict, relax, sentence-level ---------
            final_metrics = {}
            categories = entity_metrics.keys()
            for cat in categories:
                final_metrics[cat] = entity_metrics[cat]
                final_metrics[cat]['sentence-level'] = sentence_metrics[cat]['sentence-level']

            # --------- Add overall (optional) ---------
            overall_metrics = ner_instance.compute_overall_metrics_separated(eval_dict_true, eval_dict_pred, sentence_eval_dict)
            final_metrics['overall'] = overall_metrics

            # Save fold result
            model_folds_metrics.append(final_metrics)
    
    # Save model result
    model_results[model_name] = model_folds_metrics

In [21]:
model_results

{'BERT': [{'Adherence': {'strict': {'precision': 0.318,
     'recall': 0.219,
     'f-1': 0.259},
    'relax': {'precision': 0.636, 'recall': 0.438, 'f-1': 0.519},
    'sentence-level': {'precision': 0.714, 'recall': 0.484, 'f-1': 0.577}},
   'Concern': {'strict': {'precision': 0.5, 'recall': 0.625, 'f-1': 0.556},
    'relax': {'precision': 0.6, 'recall': 0.75, 'f-1': 0.667},
    'sentence-level': {'precision': 0.6, 'recall': 0.75, 'f-1': 0.667}},
   'Education': {'strict': {'precision': 0.556, 'recall': 0.625, 'f-1': 0.588},
    'relax': {'precision': 0.741, 'recall': 0.833, 'f-1': 0.784},
    'sentence-level': {'precision': 0.81, 'recall': 0.895, 'f-1': 0.85}},
   'Employment': {'strict': {'precision': 0.567,
     'recall': 0.567,
     'f-1': 0.567},
    'relax': {'precision': 0.667, 'recall': 0.667, 'f-1': 0.667},
    'sentence-level': {'precision': 0.792, 'recall': 0.76, 'f-1': 0.776}},
   'Financial': {'strict': {'precision': 0.443, 'recall': 0.482, 'f-1': 0.462},
    'relax': {'p

In [22]:
def average_metrics(fold_metrics_list):
    avg_metrics = {}
    num_folds = len(fold_metrics_list)
    
    categories = fold_metrics_list[0].keys()

    for cat in categories:
        avg_metrics[cat] = {'strict': {'precision': 0, 'recall': 0, 'f-1': 0},
                            'relax': {'precision': 0, 'recall': 0, 'f-1': 0},
                            'sentence-level': {'precision': 0, 'recall': 0, 'f-1': 0}}
        
        # Sum across folds
        for fold_metric in fold_metrics_list:
            for metric_type in ['strict', 'relax', 'sentence-level']:
                avg_metrics[cat][metric_type]['precision'] += fold_metric[cat][metric_type]['precision']
                avg_metrics[cat][metric_type]['recall'] += fold_metric[cat][metric_type]['recall']
                avg_metrics[cat][metric_type]['f-1'] += fold_metric[cat][metric_type]['f-1']
        
        # Average
        for metric_type in ['strict', 'relax', 'sentence-level']:
            avg_metrics[cat][metric_type]['precision'] = round(avg_metrics[cat][metric_type]['precision'] / num_folds, 3)
            avg_metrics[cat][metric_type]['recall'] = round(avg_metrics[cat][metric_type]['recall'] / num_folds, 3)
            avg_metrics[cat][metric_type]['f-1'] = round(avg_metrics[cat][metric_type]['f-1'] / num_folds, 3)
    
    return avg_metrics


In [23]:
final_avg_results = {}

for model_name in model_results.keys():
    avg_metrics = average_metrics(model_results[model_name])
    final_avg_results[model_name] = avg_metrics
    
with open('../output/model_performance.json','w') as f:
    json.dump(final_avg_results, f, indent=4)


In [24]:
import pandas as pd

def convert_overall_to_dataframe(final_avg_results):
    rows = []
    for model_name, metrics in final_avg_results.items():
        row = {'Model': model_name}
        for eval_type in ['strict', 'relax', 'sentence-level']:
            row[f'{eval_type}_precision'] = metrics['overall'][eval_type]['precision']
            row[f'{eval_type}_recall'] = metrics['overall'][eval_type]['recall']
            row[f'{eval_type}_f1'] = metrics['overall'][eval_type]['f-1']
        rows.append(row)
    df = pd.DataFrame(rows)
    return df
df = convert_overall_to_dataframe(final_avg_results)

# Save to Excel
df.to_excel('../output/model_overall_summary.xlsx', index=False)

In [26]:
def convert_bert_per_category_to_dataframe(final_avg_results, model_name='BioBERT'):
    rows = []
    bert_metrics = final_avg_results[model_name]
    for category, metrics in bert_metrics.items():
        if category == 'overall':  # Skip overall
            continue
        row = {'Category': category}
        for eval_type in ['strict', 'relax', 'sentence-level']:
            row[f'{eval_type}_precision'] = metrics[eval_type]['precision']
            row[f'{eval_type}_recall'] = metrics[eval_type]['recall']
            row[f'{eval_type}_f1'] = metrics[eval_type]['f-1']
        rows.append(row)
    df = pd.DataFrame(rows)
    return df
df_bert = convert_bert_per_category_to_dataframe(final_avg_results, model_name='roberta')
df_bert.to_excel('../output/roberta_per_category.xlsx', index=False)